In [1]:
pip install openai opencv-python moviepy


   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   --------- ------------------------------ 9.2/39.0 MB 47.4 MB/s eta 0:00:01
   -------------------- ------------------- 20.2/39.0 MB 53.1 MB/s eta 0:00:01
   ----------------------------------- ---- 34.3/39.0 MB 54.5 MB/s eta 0:00:01
   ---------------------------------------- 39.0/39.0 MB 46.7 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 12.6/12.6 MB 60.8 MB/s  0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ---------------------------------------- 7.0/7.0 MB 48.0 MB/s  0:00:00
   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   -------------------- ------------------- 15.7/31.2 MB 82.7 MB/s eta 0:00:01
   ------------------------------------ --- 28.3/31.2 MB 69.1 MB/s eta 0:00:01
   ---------------------------------------- 31.2/31.2 MB 56.6 MB/s  0:00:00

  Attempting uninstall: 

  You can safely remove it manually.
  You can safely remove it manually.


In [2]:
pip list

Package                                  Version
---------------------------------------- -----------
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.2
aiohttp-retry                            2.9.1
aiosignal                                1.4.0
alabaster                                1.0.0
annotated-types                          0.7.0
anthropic                                0.72.0
anyio                                    4.11.0
appdirs                                  1.4.4
argon2-cffi                              25.1.0
argon2-cffi-bindings                     25.1.0
arrow                                    1.4.0
astroid                                  4.0.2
asttokens                                3.0.0
async-lru                                2.0.5
attrs                                    25.4.0
babel                                    2.17.0
backoff                                  2.2.1
bcrypt                                   5.0.

In [3]:
import os
import cv2
import base64
import json
from openai import OpenAI

In [10]:
import os
import cv2
import base64
import json
from openai import OpenAI  # pip install openai

# ========== CONFIG ==========
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # or put your key as a string
VIDEO_PATH = "C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/cam1.mp4"
OUTPUT_PATH = "incident_fall_trimmed.mp4"
FRAME_STEP_SEC = 0.5
MODEL_NAME = "gpt-4o-mini"  # any vision-capable OpenAI model
# ============================

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """
You are an incident-detection expert for CCTV footage in elder-care homes.

Your job:
- Detect if a PERSON FALLS down (slips, trips, collapses, loses balance and ends up on the floor).
- The input is:
  - A short 15–20 second clip, but you only see sampled frames.
  - For each frame, you'll know the timestamp in seconds.

You must estimate:
- start_second: when the fall motion clearly begins
- end_second: when the fall clearly ends (person has hit the ground or is lying/sitting and the motion of falling is over)

Rules:
- If you are not reasonably sure that a real FALL occurred, return null for both start_second and end_second
  and set confidence <= 0.3.
- Ignore small posture changes, bending down, sitting intentionally, or leaning.
- Only mark as FALL when it appears unintended (loss of balance, sudden downward movement).

Output:
- Reply in STRICT JSON with the following keys only:
  {
    "start_second": float or null,
    "end_second": float or null,
    "confidence": float,
    "notes": "short reasoning in 1–2 sentences"
  }

Important:
- Use the timestamps of the frames as your reference.
- If you see the fall starting between two frames, approximate to the nearest 0.1 second.
- Never add extra keys or comments outside of the JSON.
"""


def sample_frames(video_path, step_sec=0.5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    frames = []
    t = 0.0

    while t < duration:
        frame_index = int(t * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame = cap.read()
        if not ok:
            break

        ok, buf = cv2.imencode(".jpg", frame)
        if not ok:
            t += step_sec
            continue

        jpg_bytes = buf.tobytes()
        b64 = base64.b64encode(jpg_bytes).decode("utf-8")
        frames.append({"time": float(t), "image_b64": b64})
        t += step_sec

    cap.release()
    return frames


def build_user_content(frames):
    content = []

    intro_text_lines = [
        "You are given sampled frames from a single short CCTV clip.",
        "Each frame below is labelled with its timestamp in seconds.",
        "",
        "Frames:"
    ]
    for idx, f in enumerate(frames):
        intro_text_lines.append(f"- Frame {idx}: t={f['time']:.2f} seconds")

    content.append({
        "type": "text",
        "text": "\n".join(intro_text_lines)
    })

    for idx, f in enumerate(frames):
        content.append({
            "type": "text",
            "text": f"Frame {idx}, t={f['time']:.2f} seconds"
        })
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{f['image_b64']}"
            }
        })

    return content


def call_llm_for_fall_detection(frames):
    user_content = build_user_content(frames)

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        temperature=0.1,
    )

    raw = resp.choices[0].message.content.strip()
    print("RAW LLM OUTPUT:", raw)

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        raise RuntimeError("LLM did not return valid JSON")

    return result


def trim_video_cv2(input_path, output_path, start_sec, end_sec, padding=0.3):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    s = max(0.0, float(start_sec) - padding)
    e = min(duration, float(end_sec) + padding)

    if e <= s:
        cap.release()
        raise ValueError(f"Invalid trim range: start={s}, end={e}")

    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    start_frame = int(s * fps)
    end_frame = int(e * fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    current = start_frame

    while current <= end_frame:
        ok, frame = cap.read()
        if not ok:
            break
        out.write(frame)
        current += 1

    cap.release()
    out.release()
    print(f"Trimmed video written to: {output_path} (from {s:.2f}s to {e:.2f}s)")


def process_video_for_fall(video_path=VIDEO_PATH, output_path=OUTPUT_PATH):
    frames = sample_frames(video_path, step_sec=FRAME_STEP_SEC)
    if not frames:
        raise RuntimeError("No frames sampled from video")

    result = call_llm_for_fall_detection(frames)
    print("LLM result parsed:", result)

    start = result.get("start_second")
    end = result.get("end_second")
    conf = float(result.get("confidence", 0))

    if start is None or end is None or conf < 0.6:
        print("No confident fall detected – not trimming automatically.")
        return

    trim_video_cv2(video_path, output_path, start, end)


if __name__ == "__main__":
    process_video_for_fall()


RuntimeError: Cannot open video: C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/cam1.mp4

In [5]:
import os
VIDEO_PATH = "C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/cam1.avi"

print("Exists?", os.path.exists(VIDEO_PATH))
print("Abspath:", os.path.abspath(VIDEO_PATH))


Exists? True
Abspath: C:\Users\msarav01\NXT24 Environment\Tech Bootcamp\cam1.avi


In [7]:
def debug_check_video(path):
    import os
    print("Checking path:", path)
    print("Exists?", os.path.exists(path))
    print("Abspath:", os.path.abspath(path))

    cap = cv2.VideoCapture(path)
    print("cap.isOpened():", cap.isOpened())
    if cap.isOpened():
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0
        print("✅ Video opened.")
        print("FPS:", fps)
        print("Total frames:", total_frames)
        print("Duration (sec):", duration)
    else:
        print("❌ OpenCV cannot open this video.")
    cap.release()


if __name__ == "__main__":
    debug_check_video(VIDEO_PATH)

Checking path: C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/cam1.avi
Exists? True
Abspath: C:\Users\msarav01\NXT24 Environment\Tech Bootcamp\cam1.avi
cap.isOpened(): True
✅ Video opened.
FPS: 120.0
Total frames: 1562
Duration (sec): 13.016666666666667


In [18]:
import os
import cv2
import base64
import json
from openai import OpenAI
from dotenv import load_dotenv  # pip install python-dotenv

# ========== CONFIG ==========
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # or put your key as a string

# 🔧 CHANGE THESE TO YOUR REAL PATHS
VIDEO_PATH = r"C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/cam1.avi"
OUTPUT_PATH = r"C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/incident_fall_trimmed.mp4"

FRAME_STEP_SEC = 0.5
MODEL_NAME = "gpt-4o-mini"
# ============================

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """
You are an incident-detection expert for CCTV footage in elder-care homes.

Your job:
1) Detect if a PERSON FALLS down (slips, trips, collapses, loses balance and ends up on the floor).
2) Estimate when the fall happens.
3) Classify the incident as: "none", "minor", or "major".

Definitions:
- "none": No actual fall. Maybe bending, sitting, or normal movement.
- "minor": A fall occurs but the person seems able to move afterwards, sits up or stands up, or shows signs of mild impact only.
- "major": A serious fall. Examples:
  - Person hits the floor with high impact (e.g., head or torso impact).
  - Person stays on the ground without getting up.
  - Person appears unconscious or not moving after the fall.
  - Collision with furniture/walls that looks severe.

Input:
- A short CCTV clip (15–20 seconds), but you only see sampled frames.
- Each frame has a timestamp in seconds.

You must estimate:
- start_second: when the fall motion clearly begins.
- end_second: when the fall clearly ends (person has hit the ground or is lying/sitting and the motion of falling is over).

Rules:
- If you are not reasonably sure that a real FALL occurred:
  - Use incident_severity = "none"
  - Use start_second = null and end_second = null
  - Use confidence <= 0.3
- Ignore:
  - Small posture changes
  - Bending down
  - Intentional sitting
  - Leaning on furniture or walls
- Only mark as FALL when it appears unintended:
  - Sudden downward movement
  - Loss of balance
  - Person ends up on the floor or very low.

Output format:
Reply in STRICT JSON with the following keys only:
{
  "start_second": float or null,
  "end_second": float or null,
  "confidence": float,
  "incident_severity": "none" | "minor" | "major",
  "notes": "short reasoning in 1–3 sentences"
}

Important:
- Use the timestamps of the frames as your reference.
- If you see the fall starting between two frames, approximate to the nearest 0.1 second.
- Do NOT add any extra keys or text outside of the JSON object.
"""


def sample_frames(video_path, step_sec=0.5):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    frames = []
    t = 0.0

    while t < duration:
        frame_index = int(t * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame = cap.read()
        if not ok:
            break

        ok, buf = cv2.imencode(".jpg", frame)
        if not ok:
            t += step_sec
            continue

        jpg_bytes = buf.tobytes()
        b64 = base64.b64encode(jpg_bytes).decode("utf-8")
        frames.append({"time": float(t), "image_b64": b64})
        t += step_sec

    cap.release()
    return frames


def build_user_content(frames):
    content = []

    intro_text_lines = [
        "You are given sampled frames from a single short CCTV clip.",
        "Each frame below is labelled with its timestamp in seconds.",
        "",
        "Frames:"
    ]
    for idx, f in enumerate(frames):
        intro_text_lines.append(f"- Frame {idx}: t={f['time']:.2f} seconds")

    content.append({
        "type": "text",
        "text": "\n".join(intro_text_lines)
    })

    for idx, f in enumerate(frames):
        content.append({
            "type": "text",
            "text": f"Frame {idx}, t={f['time']:.2f} seconds"
        })
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{f['image_b64']}"
            }
        })

    return content


def call_llm_for_fall_detection(frames):
    user_content = build_user_content(frames)

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        temperature=0.1,
        response_format={"type": "json_object"},  
    )

    raw = resp.choices[0].message.content.strip()
    print("RAW LLM OUTPUT:", raw)

    # Extract token usage from OpenAI metadata
    usage = resp.usage
    input_tokens = usage.prompt_tokens
    output_tokens = usage.completion_tokens

    # Calculate cost
    call_cost = record_usage(MODEL_NAME, input_tokens, output_tokens)

    print(f"🧮 Token usage → Input: {input_tokens}, Output: {output_tokens}")
    print(f"💲 Cost of this call: ${call_cost:.6f}")

    # Clean up possible ```json wrappers
    if raw.startswith("```"):
        parts = raw.split("```")
        if len(parts) >= 2:
            raw = parts[1].strip()
            if raw.lower().startswith("json"):
                raw = raw[4:].lstrip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        raise RuntimeError(f"LLM did not return valid JSON. Raw response was:\n{raw}")

    return result




def trim_video_cv2(
    input_path,
    output_path,
    start_sec,
    end_sec,
    padding_before=1.0,
    padding_after=2.0,
    min_window=3.0,
    max_window=5.0,
):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0

    # 1️⃣ Initial window: some context before + after
    s = start_sec - padding_before
    e = end_sec + padding_after

    # 2️⃣ Enforce minimum length (e.g. 3 seconds)
    if (e - s) < min_window:
        center = (start_sec + end_sec) / 2.0
        s = center - min_window / 2.0
        e = center + min_window / 2.0

    # 3️⃣ Optional: cap to maximum length (e.g. 5 seconds)
    if (e - s) > max_window:
        center = (start_sec + end_sec) / 2.0
        s = center - max_window / 2.0
        e = center + max_window / 2.0

    # 4️⃣ Clamp to video bounds
    s = max(0.0, s)
    e = min(duration, e)

    if e <= s:
        cap.release()
        raise ValueError(f"Invalid trim range: start={s}, end={e}")

    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    start_frame = int(s * fps)
    end_frame = int(e * fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    current = start_frame

    while current <= end_frame:
        ok, frame = cap.read()
        if not ok:
            break
        out.write(frame)
        current += 1

    cap.release()
    out.release()
    print(
        f"Trimmed video written to: {output_path} "
        f"(from {s:.2f}s to {e:.2f}s, length ~{e - s:.2f}s)"
    )



def process_video_for_fall(video_path=VIDEO_PATH, output_path=OUTPUT_PATH):
    frames = sample_frames(video_path, step_sec=FRAME_STEP_SEC)
    if not frames:
        raise RuntimeError("No frames sampled from video")

    result = call_llm_for_fall_detection(frames)
    print("LLM result parsed:", result)

    start = result.get("start_second")
    end = result.get("end_second")
    conf = float(result.get("confidence", 0))

    severity = result.get("incident_severity", "none")
    notes = result.get("notes", "")
    print(f"Severity: {severity}, confidence: {conf}, notes: {notes}")


    if start is None or end is None or conf < 0.6:
        print("No confident fall detected – not trimming automatically.")
        return

    trim_video_cv2(video_path, output_path, start, end)


def debug_check_video(path):
    print("Checking video path:", os.path.abspath(path))
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print("❌ OpenCV cannot open this video.")
        return
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    print("✅ Video opened.")
    print("FPS:", fps)
    print("Total frames:", total_frames)
    print("Duration (sec):", duration)
    cap.release()


# ========= LLM USAGE + COST TRACKER =========

# Pricing (update anytime if OpenAI changes)
PRICING = {
    "gpt-4o-mini": {
        "input_per_million": 0.15,    # $0.15 / 1M input tokens
        "output_per_million": 0.60    # $0.60 / 1M output tokens
    }
}

GLOBAL_USAGE = {
    "total_input_tokens": 0,
    "total_output_tokens": 0,
    "total_cost_usd": 0.0
}


def calculate_llm_cost(model_name, input_tokens, output_tokens):
    """Returns cost (USD) for a single LLM call."""
    pricing = PRICING.get(model_name)

    if not pricing:
        raise ValueError(f"No pricing info for model: {model_name}")

    cost_in = (input_tokens / 1_000_000) * pricing["input_per_million"]
    cost_out = (output_tokens / 1_000_000) * pricing["output_per_million"]

    return cost_in + cost_out


def record_usage(model_name, input_tokens, output_tokens):
    """Records global usage and returns single-call cost."""
    cost = calculate_llm_cost(model_name, input_tokens, output_tokens)

    GLOBAL_USAGE["total_input_tokens"] += input_tokens
    GLOBAL_USAGE["total_output_tokens"] += output_tokens
    GLOBAL_USAGE["total_cost_usd"] += cost

    return cost

def print_usage_summary():
    print("\n========= LLM USAGE SUMMARY =========")
    print(f"Total input tokens:   {GLOBAL_USAGE['total_input_tokens']}")
    print(f"Total output tokens:  {GLOBAL_USAGE['total_output_tokens']}")
    print(f"Total cost (USD):     ${GLOBAL_USAGE['total_cost_usd']:.6f}")
    print("=====================================\n")




if __name__ == "__main__":
    debug_check_video(VIDEO_PATH)
    process_video_for_fall()
    print_usage_summary()


Checking video path: C:\Users\msarav01\NXT24 Environment\Tech Bootcamp\cam1.avi
✅ Video opened.
FPS: 120.0
Total frames: 1562
Duration (sec): 13.016666666666667
RAW LLM OUTPUT: {
  "start_second": 9.0,
  "end_second": 10.0,
  "confidence": 0.8,
  "incident_severity": "major",
  "notes": "The person appears to lose balance and falls to the ground, remaining motionless for a brief period before lying on the floor."
}
🧮 Token usage → Input: 383625, Output: 70
💲 Cost of this call: $0.057586
LLM result parsed: {'start_second': 9.0, 'end_second': 10.0, 'confidence': 0.8, 'incident_severity': 'major', 'notes': 'The person appears to lose balance and falls to the ground, remaining motionless for a brief period before lying on the floor.'}
Severity: major, confidence: 0.8, notes: The person appears to lose balance and falls to the ground, remaining motionless for a brief period before lying on the floor.
Trimmed video written to: C:/Users/msarav01/NXT24 Environment/Tech Bootcamp/incident_fall_t